# Notebook 08: DiT Block 实现 + AdaLN-Zero 验证

**目标**：从零实现一个 DiT block，验证 AdaLN-Zero 的 identity init 性质。

**前置**：L11

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Patchify

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=2, in_ch=4, dim=384):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch_size, stride=patch_size)
        self.num_patches = (img_size // patch_size) ** 2
    def forward(self, x):
        x = self.proj(x)  # (B, dim, H/p, W/p)
        return x.flatten(2).transpose(1, 2)  # (B, N, dim)

# Sanity check
patcher = PatchEmbed(img_size=32, patch_size=2, in_ch=4, dim=384)
x = torch.randn(2, 4, 32, 32)
tokens = patcher(x)
print(f'输入: {x.shape} → tokens: {tokens.shape}')  # (2, 256, 384)

## 2. AdaLN-Zero DiT Block

In [ ]:
def modulate(x, scale, shift):
    # x: (B, N, D), scale/shift: (B, D)
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class DiTBlock(nn.Module):
    def __init__(self, dim=384, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
        # 6 个调制：γ1, β1, α1, γ2, β2, α2
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        # ZERO INIT - 这是 AdaLN-Zero 的关键
        nn.init.zeros_(self.adaLN[1].weight)
        nn.init.zeros_(self.adaLN[1].bias)
    def forward(self, x, c):
        g1, b1, a1, g2, b2, a2 = self.adaLN(c).chunk(6, dim=-1)
        # Attn branch
        h = modulate(self.norm1(x), g1, b1)
        attn_out, _ = self.attn(h, h, h)
        x = x + a1.unsqueeze(1) * attn_out
        # MLP branch
        h = modulate(self.norm2(x), g2, b2)
        x = x + a2.unsqueeze(1) * self.mlp(h)
        return x

## 3. 验证 Identity Init

AdaLN-Zero 设计的核心：训练**之前**，block 应该是 identity 函数（输出 = 输入）。

我们用一个随机条件向量去测试。

In [ ]:
block = DiTBlock(dim=384, num_heads=6)
block.eval()

x = torch.randn(2, 256, 384)
c = torch.randn(2, 384)  # 条件 embedding
y = block(x, c)

diff = (y - x).abs().mean().item()
print(f'|y - x| mean = {diff:.2e}')
assert diff < 1e-6, f'AdaLN-Zero 应该让 init block 是 identity，但 diff = {diff}'
print('✓ AdaLN-Zero identity init 验证通过')

## 4. 对比：没有 Zero Init 的 AdaLN

如果用标准初始化的 Linear，输出与输入差异较大。

In [ ]:
class StandardAdaLN(DiTBlock):
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__(dim, num_heads, mlp_ratio)
        # 用标准 Linear init（覆盖 zero init）
        nn.init.xavier_uniform_(self.adaLN[1].weight)
        nn.init.zeros_(self.adaLN[1].bias)

block_std = StandardAdaLN(dim=384, num_heads=6)
block_std.eval()

y_std = block_std(x, c)
diff_std = (y_std - x).abs().mean().item()
print(f'Standard init: |y - x| mean = {diff_std:.4f}')
print(f'Zero init:     |y - x| mean = {diff:.4e}')
print(f'\n→ Zero init 让 init 时网络输出 = 输入（identity），训练易收敛。')

## 5. 完整 DiT 模型 (Tiny)

In [ ]:
class TinyDiT(nn.Module):
    def __init__(self, img_size=32, patch_size=2, in_ch=4, dim=192, depth=6, num_heads=4):
        super().__init__()
        self.patch = PatchEmbed(img_size, patch_size, in_ch, dim)
        num_patches = self.patch.num_patches
        self.pos_emb = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        # Time + label embedding
        self.t_emb = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.y_emb = nn.Embedding(11, dim)  # 10 classes + null
        # Blocks
        self.blocks = nn.ModuleList([DiTBlock(dim, num_heads) for _ in range(depth)])
        # Final layer (also AdaLN + zero init linear)
        self.norm_final = nn.LayerNorm(dim, elementwise_affine=False)
        self.adaLN_final = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        nn.init.zeros_(self.adaLN_final[1].weight)
        self.head = nn.Linear(dim, patch_size * patch_size * in_ch)
        nn.init.zeros_(self.head.weight)  # zero init the output
        nn.init.zeros_(self.head.bias)
        self.patch_size = patch_size
        self.in_ch = in_ch
        self.img_size = img_size
    def timestep_emb(self, t, dim):
        half = dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)
    def forward(self, x, t, y):
        # x: (B, C, H, W), t: (B,), y: (B,)
        tokens = self.patch(x) + self.pos_emb
        c = self.t_emb(self.timestep_emb(t, tokens.shape[-1])) + self.y_emb(y)
        for block in self.blocks:
            tokens = block(tokens, c)
        g, b = self.adaLN_final(c).chunk(2, dim=-1)
        tokens = modulate(self.norm_final(tokens), g, b)
        out = self.head(tokens)  # (B, N, p²*C)
        # Unpatchify
        p = self.patch_size
        h = w = self.img_size // p
        out = out.reshape(out.shape[0], h, w, p, p, self.in_ch)
        out = out.permute(0, 5, 1, 3, 2, 4).reshape(-1, self.in_ch, h*p, w*p)
        return out

model = TinyDiT()
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
x = torch.randn(2, 4, 32, 32)
t = torch.randint(0, 1000, (2,))
y = torch.randint(0, 10, (2,))
out = model(x, t, y)
print(f'Forward OK: in {x.shape} → out {out.shape}')
print(f'Init output magnitude: {out.abs().mean():.4e}  (should be ~0 because of zero head init)')

## 思考题

1. 把 zero init 全去掉（用 Xavier），训练同一任务，观察 loss 收敛差异
2. AdaLN-Zero 的 α 参数（残差缩放）和 γ/β（LN 调制）哪个对训练更关键？做 ablation
3. 为什么最后的 head linear 也要 zero init？（提示：让初始预测 noise 为 0，等价于 identity output）
4. 把 MultiheadAttention 改成 RoPE 实现，看 long-context（512 tokens）下质量差异